DATASET CONFIGURATION

In [1]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch
from torch import nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os
import torch


from cesnet_datazoo.datasets import CESNET_TLS22
from cesnet_datazoo.config import DatasetConfig, AppSelection
from cesnet_datazoo.config import ScalerEnum

import random
import numpy as np

dataset = CESNET_TLS22("../datasets/CESNET-TLS22/", size="XS")
dataset_config = DatasetConfig(
    dataset=dataset,
    apps_selection=AppSelection.TOPX_KNOWN,
    apps_selection_topx=10,
    train_period_name="W-2021-40",
    test_period_name="W-2021-41",
    psizes_scaler=ScalerEnum.MINMAX,
    ipt_scaler=ScalerEnum.MINMAX,
    flowstats_scaler= ScalerEnum.MINMAX,
    train_dataloader_order= 'random',
    
    
    batch_size=32,
)

dataset.set_dataset_config_and_initialize(dataset_config)

train_dl = dataset.get_train_dataloader()
val_dl = dataset.get_val_dataloader()
test_dl = dataset.get_test_dataloader()



SSL MODEL

In [2]:
class Classifier1(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()

        # Encoder: Transforms the input feature vector into a lower-dimensional space
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            
        )

        # Classifier: Outputs a logit for each class
        self.classifier = nn.Sequential(
            nn.Linear(512, output_size),
            
        )

    def forward(self, x):
        encoded = self.encoder(x)
        logits = self.classifier(encoded)
        return logits
    def extract_embeddings(self,x):
        encoded = self.encoder(x)
        return encoded


FUNCTIONS FOR CONTRASTIVE LEARNING

In [3]:
def l2_normalize(embeddings):
    """Applies L2 normalization to embeddings."""
    return embeddings / (embeddings.norm(dim=1, keepdim=True) + 1e-12)

def infoNCE_loss(anchor_embeddings, positive_embeddings, negative_embeddings, temperature=0.95):
    """
    Calculates the InfoNCE loss with normalized embeddings and separate handling
    for positive and negative similarities.

    Args:
    - anchor_embeddings: Embeddings of the original sequences.
    - positive_embeddings: Embeddings of the masked versions of the original sequences.
    - negative_embeddings: Embeddings of masked versions of different sequences.
    - temperature: A scaling factor to manage the sharpness of the distribution.

    Returns:
    - The calculated InfoNCE loss.
    """
    # Normalize embeddings
    anchor_embeddings = l2_normalize(anchor_embeddings)
    positive_embeddings = l2_normalize(positive_embeddings)
    negative_embeddings = l2_normalize(negative_embeddings)

    # Calculate positive similarity
     # InfoNCE loss
    positive_term = torch.sum(anchor_embeddings * positive_embeddings, dim=1) / temperature
    negative_term = torch.logsumexp(torch.matmul(anchor_embeddings, negative_embeddings.T) / temperature, dim=1)



    loss = -positive_term + negative_term

    return loss.mean()


In [4]:
def select_negative_pairs_within_batch(batch_sequences, batch_labels):
    """
    For each sequence in the batch, select a negative sequence from a different app label.
    Args:
    - batch_sequences: Tensor of shape [batch_size, seq_len, num_features]
    - batch_labels: Tensor of shape [batch_size] with app labels for each sequence
    
    Returns:
    - negative_indices: Indices of selected negative sequences within the batch
    """
    batch_size = batch_sequences.size(0)
    negative_indices = torch.zeros(batch_size, dtype=torch.long)
    
    for i in range(batch_size):
        different_labels_mask = batch_labels != batch_labels[i]
        different_labels_indices = torch.where(different_labels_mask)[0]
        selected_negative = different_labels_indices[torch.randint(len(different_labels_indices), (1,))]
        negative_indices[i] = selected_negative
    device = batch_sequences.device
    return negative_indices.to(device)

In [5]:
def select_positive_pairs_within_batch(batch_sequences, batch_labels):
    """
    For each sequence in the batch, select a negative sequence from a different app label.
    Args:
    - batch_sequences: Tensor of shape [batch_size, seq_len, num_features]
    - batch_labels: Tensor of shape [batch_size] with app labels for each sequence
    
    Returns:
    - negative_indices: Indices of selected negative sequences within the batch
    """
    batch_size = batch_sequences.size(0)
    positive_indices = torch.zeros(batch_size, dtype=torch.long)
    
    for i in range(batch_size):
        same_labels_mask = batch_labels == batch_labels[i]
        same_labels_mask = torch.where(same_labels_mask)[0]
        selected_positive = same_labels_mask[torch.randint(len(same_labels_mask), (1,))]
        positive_indices[i] = selected_positive
    device = batch_sequences.device
    return positive_indices.to(device)

In [6]:
def print_cosine_similarity(embedding1, embedding2):
    # Ensure the embeddings are floats for torch.matmul operation
    embedding1 = embedding1.float()
    embedding2 = embedding2.float()
    
    # Normalize the embeddings to unit vectors
    embedding1_norm = F.normalize(embedding1, p=2, dim=0)
    embedding2_norm = F.normalize(embedding2, p=2, dim=0)
    
    # Calculate cosine similarity as dot product of the normalized vectors
    cosine_sim = torch.dot(embedding1_norm, embedding2_norm)
    
    print(f"Cosine Similarity: {cosine_sim.item()}")

CUSTOM LOSS FUNCTION

In [7]:
def class_loss(predictions, ground_truth):
    """
    Custom loss function that calculates loss for each sample individually and then computes the mean.
    
    Args:
    - predictions: Tensor of shape [batch_size, n_classes], with raw scores for each class.
    - ground_truth: Tensor of shape [batch_size], with the indices of the correct class for each sample.
    
    Returns:
    - loss: Scalar tensor representing the mean loss value across the batch.
    """
    # Initialize a tensor to store individual losses
    individual_losses = torch.zeros(predictions.size(0))
    
    for i in range(predictions.size(0)):
        # Compute the loss for each sample individually
        individual_loss = F.cross_entropy(predictions[i].unsqueeze(0), ground_truth[i].unsqueeze(0))
        individual_losses[i] = individual_loss
    
    # Compute the mean of the individual losses
    mean_loss = torch.mean(individual_losses)
    
    return mean_loss


SSL TRAINING LOOP 

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = np.arange(0, 1.1, 0.1)

for class_weight in weights:
    contrastive_weight = 1 - class_weight  
    classifier = Classifier1(input_size=90, output_size=10).to(device)
    model_name = f'model_classifier={class_weight:.1f}_contrastive={contrastive_weight:.1f}.pth'
    print(model_name)

    classifier.to(device)

    # Loss function

    # Optimizer
    optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)

    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=5, verbose=True)

    # Training parameters
    num_epochs = 30
    patience = 2
    best_val_loss = float('inf')
    epochs_without_improvement = 0




    for epoch in range(num_epochs):
        classifier.train()
        train_losses = []

        for i, (_, ppi,_,labels) in enumerate(train_dl):
            
            
            optimizer.zero_grad()
            ppi = torch.from_numpy(ppi).to(device)
            ppi = ppi.flatten(start_dim=-2)
            labels = torch.from_numpy(labels).to(device)
            
            
            
            logits = classifier(ppi)
            logits = logits.squeeze(0)
            loss_class = class_loss(logits, labels)
          
            negative_indices = select_negative_pairs_within_batch(ppi, labels)
            negative_sequences = ppi[negative_indices]

            positive_indices = select_positive_pairs_within_batch(ppi, labels)
            positive_sequences = ppi[positive_indices]

             
            
            anchor_embeddings = classifier.extract_embeddings(ppi)
            positive_embeddings = classifier.extract_embeddings(positive_sequences)
            negative_embeddings = classifier.extract_embeddings(negative_sequences)
            
            contrastive_loss = infoNCE_loss(anchor_embeddings,positive_embeddings,negative_embeddings)
            
    

            loss = contrastive_weight*contrastive_loss + class_weight* loss_class
            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())

            if i % 1000 == 0:
                print(f"Epoch {epoch} Batch {i}, Loss: {loss.item()} ")
                print_cosine_similarity(anchor_embeddings[0],positive_embeddings[0])
                print_cosine_similarity(anchor_embeddings[0],negative_embeddings[0])


        # Validation phase
        classifier.eval()
        all_preds = []
        all_labels = []
        total_accuracy = 0
        batches = 0
        val_losses = []
        with torch.no_grad():
            for i, (_, ppi,_,labels) in enumerate(val_dl):
                
                ppi = torch.from_numpy(ppi).to(device)
                ppi = ppi.flatten(start_dim = -2)
                
                labels = torch.from_numpy(labels).to(device)
                negative_indices = select_negative_pairs_within_batch(ppi, labels)
                negative_sequences = ppi[negative_indices]

                positive_indices = select_positive_pairs_within_batch(ppi, labels)
                positive_sequences = ppi[positive_indices]
                
                anchor_embeddings = classifier.extract_embeddings(ppi)
                positive_embeddings = classifier.extract_embeddings(positive_sequences)
                negative_embeddings = classifier.extract_embeddings(negative_sequences)

                contrastive_loss_val = infoNCE_loss(anchor_embeddings,positive_embeddings,negative_embeddings)

                
                
                logits = classifier(ppi)
                logits = logits.squeeze(0)
                
                class_loss_val = class_loss(logits, labels)
                
                val_loss = contrastive_weight*contrastive_loss + class_weight* loss_class
                val_losses.append(val_loss.item())
                predicted_labels = torch.argmax(logits, dim=1)
                apps = dataset.get_known_apps()
                batches +=1
    
            
            
                predicted_apps = [apps[idx] for idx in predicted_labels]
                actual_apps = [apps[idx] for idx in labels]

                
                all_preds.extend(predicted_labels.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
                # Compare predicted labels with actual labels
                correct_predictions = (predicted_labels == labels).sum().item()
                total_predictions = labels.size(0)
                accuracy = correct_predictions / total_predictions
                total_accuracy += accuracy
            
            if i % 1000 == 0:
                print(f"Batch {i}, loss: {loss.item()}")
                print(f"Predicted labels: {predicted_apps}")
                print(f"Actual labels: {actual_apps}\n")
                print(f"Accuracy: {accuracy * 100:.2f}%\n")            

            if i % 1000 == 0:
                print(f"Validation - Batch {i}, Loss: {val_loss.item()}")
                print(f"Predicted labels: {predicted_apps}")
                print(f"Actual labels: {actual_apps}\n")
                

        avg_val_loss = np.mean(val_losses)
        print(f"Epoch [{epoch+1}/{num_epochs}], Average Validation Loss: {avg_val_loss}")
        print(f"Validation - Batch {i}, Loss: {val_loss.item()}")
        
        print(f"Average accuracy: {total_accuracy/batches * 100:.2f}%")

        scheduler.step(avg_val_loss)

        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            torch.save(classifier.state_dict(), model_name)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Stopping early at epoch {epoch+1}. No improvement in validation loss for {patience} consecutive epochs. Best Loss: {best_val_loss:.4f}.")
                break

    # Evaluate model on validation set again or perform further analysis as needed
    # Load the best model
    print(f"Average accuracy: {total_accuracy/batches * 100:.2f}%")
    del classifier, optimizer, scheduler
    torch.cuda.empty_cache()


model_classifier=0.0_contrastive=1.0.pth
Epoch 0 Batch 0, Loss: 3.3671295642852783 
Cosine Similarity: 0.9999998807907104
Cosine Similarity: 0.7463052272796631
Epoch 0 Batch 1000, Loss: 3.0322651863098145 
Cosine Similarity: 1.0000001192092896
Cosine Similarity: 0.0
Epoch 0 Batch 2000, Loss: 3.0820789337158203 
Cosine Similarity: 0.3161761462688446
Cosine Similarity: 0.0
Epoch 0 Batch 3000, Loss: 2.997997522354126 
Cosine Similarity: 0.9441661834716797
Cosine Similarity: 0.0
Epoch 0 Batch 4000, Loss: 3.199531316757202 
Cosine Similarity: 0.0
Cosine Similarity: 0.0
Epoch 0 Batch 5000, Loss: 2.968364953994751 
Cosine Similarity: 1.0
Cosine Similarity: 0.0
Epoch 0 Batch 6000, Loss: 3.053710460662842 
Cosine Similarity: 1.0
Cosine Similarity: 0.0
Epoch 0 Batch 7000, Loss: 3.074882984161377 
Cosine Similarity: 1.0
Cosine Similarity: 0.0
Epoch 0 Batch 8000, Loss: 3.1250391006469727 
Cosine Similarity: 1.0
Cosine Similarity: 0.0
Epoch 0 Batch 9000, Loss: 3.0662436485290527 
Cosine Similarity:

FEATURE EXTRACTION AND DATASET CREATION

In [10]:
from torch.utils.data import Dataset
import os
import torch

class NetFlowDataset(Dataset):
    def __init__(self, features_dir, labels_dir):
        self.features_files = [os.path.join(features_dir, f) for f in sorted(os.listdir(features_dir))]
        self.labels_files = [os.path.join(labels_dir, f) for f in sorted(os.listdir(labels_dir))]
        
        assert len(self.features_files) == len(self.labels_files), "Mismatched number of features and labels files."

    def __len__(self):
        return len(self.features_files)

    def __getitem__(self, idx):
        features = torch.load(self.features_files[idx])
        labels = torch.load(self.labels_files[idx])
        return features, labels


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = np.arange(0, 1.1, 0.1)
for class_weight in weights:
    contrastive_weight = 1 - class_weight  
    classifier = Classifier1(input_size=90, output_size=10).to(device)
    model_name = f'model_classifier={class_weight:.1f}_contrastive={contrastive_weight:.1f}.pth'

    # Assuming the model and data_loader are already defined and initialized as per your setup
    classifier.load_state_dict(torch.load(model_name))
    classifier.eval()

    # Directory setup
    features_dir = f'extracted_features_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}'
    labels_dir = f'extracted_labels_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}'
    os.makedirs(features_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    with torch.no_grad():
        for i, (_,ppi, additional_features, labels) in enumerate(train_dl):
            # Data pre-processing steps you mentioned
            ppi = torch.from_numpy(ppi).to(device)
            labels = torch.from_numpy(labels).to(device)
            flattened_ppi = ppi.flatten(start_dim=-2).to(device)     

            embeddings = classifier.extract_embeddings(flattened_ppi)

            concatenated_features = torch.cat((flattened_ppi, embeddings), dim=1)
            
            # Save the features and labels to disk
            features_path = os.path.join(features_dir, f'features_batch_{i}.pt')
            labels_path = os.path.join(labels_dir, f'labels_batch_{i}.pt')
            torch.save(concatenated_features, features_path)
            torch.save(labels, labels_path)

            if i % 1000 == 0:
                print(f"Batch {i}, features saved to {features_path}, labels saved to {labels_path}")





    
    classifier = Classifier1(input_size=90, output_size=10).to(device)
    model_name = f'model_classifier={class_weight:.1f}_contrastive={contrastive_weight:.1f}.pth'
    # Assuming the model and data_loader are already defined and initialized as per your setup
    classifier.load_state_dict(torch.load(model_name))
    classifier.eval()
    
    # Directory setup
    features_dir = f'extracted_features_val_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}'
    labels_dir = f'extracted_labels_val_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}'
    
    os.makedirs(features_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    with torch.no_grad():
        for i, (_,ppi, additional_features, labels) in enumerate(val_dl):
            # Data pre-processing steps you mentioned
            ppi = torch.from_numpy(ppi).to(device)
            labels = torch.from_numpy(labels).to(device)
            flattened_ppi = ppi.flatten(start_dim=-2).to(device)     

            embeddings = classifier.extract_embeddings(flattened_ppi)

            concatenated_features = torch.cat((flattened_ppi, embeddings), dim=1)
            
            # Save the features and labels to disk
            features_path = os.path.join(features_dir, f'features_batch_{i}.pt')
            labels_path = os.path.join(labels_dir, f'labels_batch_{i}.pt')
            torch.save(concatenated_features, features_path)
            torch.save(labels, labels_path)

            if i % 1000 == 0:
                print(f"Batch {i}, features saved to {features_path}, labels saved to {labels_path}")


    

    classifier = Classifier1(input_size=90, output_size=10).to(device)
    model_name = f'model_classifier={class_weight:.1f}_contrastive={contrastive_weight:.1f}.pth'
    # Assuming the model and data_loader are already defined and initialized as per your setup
    classifier.load_state_dict(torch.load(model_name))
    classifier.eval()
    
    # Directory setup
    features_dir = f'extracted_features_test_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}'
    labels_dir = f'extracted_labels_test_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}'
    
    os.makedirs(features_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    with torch.no_grad():
        for i, (_,ppi, additional_features, labels) in enumerate(test_dl):
            # Data pre-processing steps you mentioned
            ppi = torch.from_numpy(ppi).to(device)
            labels = torch.from_numpy(labels).to(device)
            flattened_ppi = ppi.flatten(start_dim=-2).to(device)     

            embeddings = classifier.extract_embeddings(flattened_ppi)

            concatenated_features = torch.cat((flattened_ppi, embeddings), dim=1)
            
            # Save the features and labels to disk
            features_path = os.path.join(features_dir, f'features_batch_{i}.pt')
            labels_path = os.path.join(labels_dir, f'labels_batch_{i}.pt')
            torch.save(concatenated_features, features_path)
            torch.save(labels, labels_path)

            if i % 1000 == 0:
                print(f"Batch {i}, features saved to {features_path}, labels saved to {labels_path}")

   


Batch 0, features saved to extracted_features_model_class=0.0_contr=1.0/features_batch_0.pt, labels saved to extracted_labels_model_class=0.0_contr=1.0/labels_batch_0.pt
Batch 1000, features saved to extracted_features_model_class=0.0_contr=1.0/features_batch_1000.pt, labels saved to extracted_labels_model_class=0.0_contr=1.0/labels_batch_1000.pt
Batch 2000, features saved to extracted_features_model_class=0.0_contr=1.0/features_batch_2000.pt, labels saved to extracted_labels_model_class=0.0_contr=1.0/labels_batch_2000.pt
Batch 3000, features saved to extracted_features_model_class=0.0_contr=1.0/features_batch_3000.pt, labels saved to extracted_labels_model_class=0.0_contr=1.0/labels_batch_3000.pt


Batch 4000, features saved to extracted_features_model_class=0.0_contr=1.0/features_batch_4000.pt, labels saved to extracted_labels_model_class=0.0_contr=1.0/labels_batch_4000.pt
Batch 5000, features saved to extracted_features_model_class=0.0_contr=1.0/features_batch_5000.pt, labels saved to extracted_labels_model_class=0.0_contr=1.0/labels_batch_5000.pt
Batch 6000, features saved to extracted_features_model_class=0.0_contr=1.0/features_batch_6000.pt, labels saved to extracted_labels_model_class=0.0_contr=1.0/labels_batch_6000.pt
Batch 7000, features saved to extracted_features_model_class=0.0_contr=1.0/features_batch_7000.pt, labels saved to extracted_labels_model_class=0.0_contr=1.0/labels_batch_7000.pt
Batch 8000, features saved to extracted_features_model_class=0.0_contr=1.0/features_batch_8000.pt, labels saved to extracted_labels_model_class=0.0_contr=1.0/labels_batch_8000.pt
Batch 9000, features saved to extracted_features_model_class=0.0_contr=1.0/features_batch_9000.pt, labels

MODEL FOR CLASSIFICATION

In [12]:
class Classifier2(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()

        # Encoder: Transforms the input feature vector into a lower-dimensional space
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            

            
           
        )

        # Classifier: Outputs a logit for each class
        self.classifier = nn.Sequential(
            nn.Linear(512, output_size),
           
        )

    def forward(self, x):
        encoded = self.encoder(x)
        logits = self.classifier(encoded)
        return logits



DOWNSTREAM TASK

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Assuming Classifier is defined
# Assuming dataloader_features and dataloader_features_val are defined
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = np.arange(0, 1.1, 0.1)
for class_weight in weights:
    contrastive_weight = 1 - class_weight
    dataset_features = NetFlowDataset(f'extracted_features_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}', f'extracted_labels_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}')

    # Initialize DataLoader
    dataloader_features = DataLoader(dataset_features, batch_size=1, shuffle=True)
        
    
    dataset_features_val = NetFlowDataset(f'extracted_features_val_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}', f'extracted_labels_val_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}')

    dataloader_features_val = DataLoader(dataset_features_val, batch_size=1, shuffle=True)
        
    dataset_features_test = NetFlowDataset(f'extracted_features_test_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}', f'extracted_labels_test_model_class={class_weight:.1f}_contr={contrastive_weight:.1f}')


    dataloader_features_test = DataLoader(dataset_features_test, batch_size=1, shuffle=True)



    
    classifier = Classifier2(input_size=602, output_size=10).to(device)
    criterion = nn.CrossEntropyLoss()  # Appropriate for classification tasks
    optimizer = optim.Adam(classifier.parameters(), lr=0.001)

    # Learning rate scheduler
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=2, verbose=True)

    num_epochs = 30
    patience = 2
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    print(model_name)
#################################TRAIN############################################
    for epoch in range(num_epochs):
        classifier.train()
        train_losses = []

        for i, (features, labels) in enumerate(dataloader_features):
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            logits = classifier(features)
            logits = logits.squeeze(0)
            labels = labels.flatten(start_dim= -2)
            loss = class_loss(logits, labels)

            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())

            if i % 1000 == 0:
                print(f"Epoch {epoch} Batch {i}, Loss: {loss.item()}")

#################################EVAL############################################
        classifier.eval()
        all_preds = []
        all_labels = []
        total_accuracy = 0
        batches = 0
        val_losses = []
        with torch.no_grad():
            for i, (features, labels) in enumerate(dataloader_features_val):
                features, labels = features.to(device), labels.to(device)
                logits = classifier(features)
                logits = logits.squeeze(0)
                labels = labels.flatten(start_dim= -2)
                
                val_loss = class_loss(logits, labels)
                val_losses.append(val_loss.item())
                predicted_labels = torch.argmax(logits, dim=1)
                apps = dataset.get_known_apps()
                batches +=1
    
            
            
                predicted_apps = [apps[idx] for idx in predicted_labels]
                actual_apps = [apps[idx] for idx in labels]

                
                all_preds.extend(predicted_labels.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
                            # Compare predicted labels with actual labels
                correct_predictions = (predicted_labels == labels).sum().item()
                total_predictions = labels.size(0)
                accuracy = correct_predictions / total_predictions
                total_accuracy += accuracy

                

        avg_val_loss = np.mean(val_losses)
        print(f"Epoch [{epoch+1}/{num_epochs}], Average Validation Loss: {avg_val_loss}")
        print(f"Validation - Batch {i}, Loss: {val_loss.item()}")
        
        print(f"Average accuracy: {total_accuracy/batches * 100:.2f}%")
        print(model_name)

        scheduler.step(avg_val_loss)

        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            torch.save(classifier.state_dict(), 'best_classifier.pth')
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Stopping early at epoch {epoch+1}. No improvement in validation loss for {patience} consecutive epochs. Best Loss: {best_val_loss:.4f}.")
                break

    # Evaluate model on validation set again or perform further analysis as needed
    # Load the best model
    print(f"Average accuracy: {total_accuracy/batches * 100:.2f}%")
   


##################################TEST#######################################################

    from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    # Load the best saved model weights
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    classifier = Classifier2(input_size=602, output_size=10).to(device)
    classifier.load_state_dict(torch.load('best_classifier.pth'))
    classifier.eval()  # Set the model to evaluation mode

    test_losses = []
    all_preds_test = []
    all_labels_test = []
    total_accuracy_test = 0
    batches = 0

    with torch.no_grad():
        for i, (features, labels) in enumerate(dataloader_features_test):
            features, labels = features.to(device), labels.to(device)
            logits = classifier(features)
            logits = logits.squeeze(0)
            labels = labels.flatten(start_dim= -2)
            test_loss = class_loss(logits, labels)
            
            
            test_losses.append(test_loss.item())

            predicted_labels = torch.argmax(logits, dim=1)
            all_preds_test.extend(predicted_labels.cpu().numpy())
            all_labels_test.extend(labels.cpu().numpy())

            # Accuracy calculation
            correct_predictions = (predicted_labels == labels).sum().item()
            total_predictions = labels.size(0)
            accuracy = correct_predictions / total_predictions
            total_accuracy_test += accuracy
            batches += 1
           

    avg_test_loss = np.mean(test_losses)
    print(f"Average Test Loss: {avg_test_loss}")
    print(f"Average Test Accuracy: {total_accuracy_test/batches * 100:.2f}%")



    # Precision, Recall, and F1 Score
    precision = precision_score(all_labels_test, all_preds_test, average='weighted')
    recall = recall_score(all_labels_test, all_preds_test, average='weighted')
    f1 = f1_score(all_labels_test, all_preds_test, average='weighted')

    print(f'Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')

    # Confusion Matrix
    """  cm = confusion_matrix(all_labels_test, all_preds_test)
    plt.figure(figsize=(30,30))
    sns.heatmap(cm, annot=True, fmt="d", cmap='Blues')
    plt.ylabel('Actual Labels')
    plt.xlabel('Predicted Labels')
    plt.title('Confusion Matrix')
    plt.show() """
    del classifier, optimizer, scheduler
    torch.cuda.empty_cache()

model_classifier=1.0_contrastive=0.0.pth
Epoch 0 Batch 0, Loss: 2.302320957183838
Epoch 0 Batch 1000, Loss: 0.6503148674964905
Epoch 0 Batch 2000, Loss: 0.6862921714782715
Epoch 0 Batch 3000, Loss: 0.482975572347641
Epoch 0 Batch 4000, Loss: 0.1825874149799347
Epoch 0 Batch 5000, Loss: 0.4237762689590454
Epoch 0 Batch 6000, Loss: 0.32041728496551514
Epoch 0 Batch 7000, Loss: 0.19805416464805603
Epoch 0 Batch 8000, Loss: 0.2768467366695404
Epoch 0 Batch 9000, Loss: 0.3193444311618805
Epoch 0 Batch 10000, Loss: 0.4417158365249634
Epoch 0 Batch 11000, Loss: 0.4656703472137451
Epoch 0 Batch 12000, Loss: 0.253898024559021
Epoch 0 Batch 13000, Loss: 0.2706422805786133
Epoch 0 Batch 14000, Loss: 0.19132603704929352
Epoch 0 Batch 15000, Loss: 0.41795721650123596
Epoch 0 Batch 16000, Loss: 0.42943495512008667
Epoch 0 Batch 17000, Loss: 0.4639870226383209
Epoch 0 Batch 18000, Loss: 0.2837325632572174
Epoch 0 Batch 19000, Loss: 0.31561973690986633
Epoch 0 Batch 20000, Loss: 0.2330201417207718
Epo

/root/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch 0 Batch 1000, Loss: 0.20789086818695068
Epoch 0 Batch 2000, Loss: 0.4826934337615967
Epoch 0 Batch 3000, Loss: 0.22157275676727295
Epoch 0 Batch 4000, Loss: 0.3758123517036438
Epoch 0 Batch 5000, Loss: 0.12644827365875244
Epoch 0 Batch 6000, Loss: 0.06404192745685577
Epoch 0 Batch 7000, Loss: 0.2025764286518097
Epoch 0 Batch 8000, Loss: 0.3104616403579712
Epoch 0 Batch 9000, Loss: 0.5628666877746582
Epoch 0 Batch 10000, Loss: 0.022723354399204254
Epoch 0 Batch 11000, Loss: 0.2805187702178955
Epoch 0 Batch 12000, Loss: 0.12581650912761688
Epoch 0 Batch 13000, Loss: 0.08069849759340286
Epoch 0 Batch 14000, Loss: 0.11829499900341034
Epoch 0 Batch 15000, Loss: 0.18237996101379395
Epoch 0 Batch 16000, Loss: 0.21495409309864044
Epoch 0 Batch 17000, Loss: 0.21863365173339844
Epoch 0 Batch 18000, Loss: 0.3662523925304413
Epoch 0 Batch 19000, Loss: 0.1345851719379425
Epoch 0 Batch 20000, Loss: 0.039975494146347046
Epoch 0 Batch 21000, Loss: 0.05936480313539505
Epoch [1/30], Average Valida

/root/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch 0 Batch 1000, Loss: 0.40972021222114563
Epoch 0 Batch 2000, Loss: 0.5624934434890747
Epoch 0 Batch 3000, Loss: 0.22937078773975372
Epoch 0 Batch 4000, Loss: 0.25963523983955383
Epoch 0 Batch 5000, Loss: 0.13116355240345
Epoch 0 Batch 6000, Loss: 0.4888324737548828
Epoch 0 Batch 7000, Loss: 0.32016390562057495
Epoch 0 Batch 8000, Loss: 0.13253557682037354
Epoch 0 Batch 9000, Loss: 0.17153744399547577
Epoch 0 Batch 10000, Loss: 0.6069042086601257
Epoch 0 Batch 11000, Loss: 0.02918691746890545
Epoch 0 Batch 12000, Loss: 0.3886902928352356
Epoch 0 Batch 13000, Loss: 0.38168761134147644
Epoch 0 Batch 14000, Loss: 0.19061881303787231
Epoch 0 Batch 15000, Loss: 0.06424117088317871
Epoch 0 Batch 16000, Loss: 0.45309579372406006
Epoch 0 Batch 17000, Loss: 0.279354453086853
Epoch 0 Batch 18000, Loss: 0.4203067719936371
Epoch 0 Batch 19000, Loss: 0.16315144300460815
Epoch 0 Batch 20000, Loss: 0.46685081720352173
Epoch 0 Batch 21000, Loss: 0.2767579257488251
Epoch [1/30], Average Validation 

...

Epoch 0 Batch 1000, Loss: 0.07559231668710709
Epoch 0 Batch 2000, Loss: 0.32169318199157715
Epoch 0 Batch 3000, Loss: 0.08722013235092163
Epoch 0 Batch 4000, Loss: 0.4383756220340729
Epoch 0 Batch 5000, Loss: 0.2524866461753845
Epoch 0 Batch 6000, Loss: 0.4777468144893646
Epoch 0 Batch 7000, Loss: 0.1409500539302826
Epoch 0 Batch 8000, Loss: 0.13653209805488586
Epoch 0 Batch 9000, Loss: 0.11514953523874283
Epoch 0 Batch 10000, Loss: 0.14101284742355347
Epoch 0 Batch 11000, Loss: 0.11299949884414673
Epoch 0 Batch 12000, Loss: 0.37642478942871094
Epoch 0 Batch 13000, Loss: 0.2412930130958557
Epoch 0 Batch 14000, Loss: 0.1867535263299942
Epoch 0 Batch 15000, Loss: 0.7404079437255859
Epoch 0 Batch 16000, Loss: 0.10727866739034653
Epoch 0 Batch 17000, Loss: 0.1190909817814827
Epoch 0 Batch 18000, Loss: 0.09073382616043091
Epoch 0 Batch 19000, Loss: 0.08593234419822693
Epoch 0 Batch 20000, Loss: 0.10079216212034225
Epoch 0 Batch 21000, Loss: 0.24309560656547546
Epoch [1/30], Average Validati

/root/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch 0 Batch 1000, Loss: 0.3872670829296112
Epoch 0 Batch 2000, Loss: 0.10546963661909103
Epoch 0 Batch 3000, Loss: 0.515393853187561
Epoch 0 Batch 4000, Loss: 0.18657130002975464
Epoch 0 Batch 5000, Loss: 0.11428256332874298
Epoch 0 Batch 6000, Loss: 0.33865028619766235
Epoch 0 Batch 7000, Loss: 0.10199050605297089
Epoch 0 Batch 8000, Loss: 0.038952816277742386
Epoch 0 Batch 9000, Loss: 0.07420635223388672
Epoch 0 Batch 10000, Loss: 0.12884780764579773
Epoch 0 Batch 11000, Loss: 0.17726188898086548
Epoch 0 Batch 12000, Loss: 0.0308344978839159
Epoch 0 Batch 13000, Loss: 0.24840356409549713
Epoch 0 Batch 14000, Loss: 0.40060433745384216
Epoch 0 Batch 15000, Loss: 0.18576496839523315
Epoch 0 Batch 16000, Loss: 0.027455227449536324
Epoch 0 Batch 17000, Loss: 0.18754632771015167
Epoch 0 Batch 18000, Loss: 0.2353832870721817
Epoch 0 Batch 19000, Loss: 0.06015868857502937
Epoch 0 Batch 20000, Loss: 0.2729858160018921
Epoch 0 Batch 21000, Loss: 0.3318147659301758
Epoch [1/30], Average Valid

/root/miniconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


Epoch 0 Batch 1000, Loss: 0.39109230041503906
Epoch 0 Batch 2000, Loss: 0.23222598433494568
Epoch 0 Batch 3000, Loss: 0.06256263703107834
Epoch 0 Batch 4000, Loss: 0.05640200525522232
Epoch 0 Batch 5000, Loss: 0.4317682981491089
Epoch 0 Batch 6000, Loss: 0.1081635057926178
Epoch 0 Batch 7000, Loss: 0.1692771464586258
Epoch 0 Batch 8000, Loss: 0.05149209126830101
Epoch 0 Batch 9000, Loss: 0.28981736302375793
Epoch 0 Batch 10000, Loss: 0.208849236369133
Epoch 0 Batch 11000, Loss: 0.09698580205440521
Epoch 0 Batch 12000, Loss: 0.14910294115543365
Epoch 0 Batch 13000, Loss: 0.1168503612279892
Epoch 0 Batch 14000, Loss: 0.12342849373817444
Epoch 0 Batch 15000, Loss: 0.06262631714344025
Epoch 0 Batch 16000, Loss: 0.11902792751789093
Epoch 0 Batch 17000, Loss: 0.036904335021972656
Epoch 0 Batch 18000, Loss: 0.07836750149726868
Epoch 0 Batch 19000, Loss: 0.022564386948943138
Epoch 0 Batch 20000, Loss: 0.08354632556438446
Epoch 0 Batch 21000, Loss: 0.11288496106863022
Epoch [1/30], Average Vali